## Week 2 Deep Research Assistant exercise.

This is an AI-powered job search assistant that helps users find relevant job opportunities based on a simple query (e.g., "Junior Data Analyst remote").

The system uses a multi-agent architecture to:
- Generate targeted job search queries
- Search the web for real job listings
- Extract structured job data (title, company, location, link, etc.)
- Format results into a clean, readable report
- Optionally send job matches via email

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional


class JobListing(BaseModel):
    title: str = Field(description="Job title")
    company: str = Field(description="Company name")
    location: str = Field(description="Job location")
    description: str = Field(description="Short job description")
    url: str = Field(description="Application link")
    salary: Optional[str] = Field(default=None, description="Salary if available")


class JobSearchResults(BaseModel):
    jobs: List[JobListing] = Field(description="List of job listings")

In [ ]:
from pydantic import BaseModel, Field
from agents import Agent

INSTRUCTIONS = (
    "You are a job search planner. Given a user's desired job role, generate targeted web search queries "
    "to find relevant job listings.\n\n"
    "Include variations such as:\n"
    "- Different job boards (LinkedIn, Indeed, Glassdoor)\n"
    "- Remote vs onsite\n"
    "- Junior, mid, senior levels\n"
    "- Location variations\n\n"
    "Generate between 5 and 8 high-quality job search queries."
)


class WebSearchItem(BaseModel):
    reason: str = Field(description="Why this search is useful")
    query: str = Field(description="Search query")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem]


planner_agent = Agent(
    name="JobPlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [ ]:
from agents import Agent, WebSearchTool, ModelSettings
from job_models import JobSearchResults

INSTRUCTIONS = (
    "You are a job search assistant. Given a job search query, search the web and extract job listings.\n\n"
    "Return structured job data including:\n"
    "- Job title\n"
    "- Company name\n"
    "- Location\n"
    "- Short description\n"
    "- Application URL\n"
    "- Salary (if available)\n\n"
    "Rules:\n"
    "- Only include real job listings\n"
    "- Avoid duplicates\n"
    "- Prefer recent postings\n"
    "- Focus on relevance to the query\n"
)

search_agent = Agent(
    name="JobSearchAgent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="medium")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
    output_type=JobSearchResults,
)

In [ ]:
from pydantic import BaseModel, Field
from agents import Agent
from typing import List

INSTRUCTIONS = (
    "You are a job assistant. Given a list of job listings, format them into a clean, structured report.\n\n"
    "Requirements:\n"
    "- Group similar roles together\n"
    "- Highlight key details\n"
    "- Keep it concise and scannable\n\n"
    "Also include:\n"
    "- A short summary\n"
    "- Practical tips for applying\n"
)


class JobReport(BaseModel):
    summary: str = Field(description="Short summary of job matches")
    markdown_report: str = Field(description="Formatted job listings in markdown")
    follow_up_suggestions: List[str] = Field(description="Suggestions for further job searching")


writer_agent = Agent(
    name="JobWriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=JobReport,
)

In [ ]:
import os
from typing import Dict

import sendgrid
from sendgrid.helpers.mail import Email, Mail, Content, To
from agents import Agent, function_tool


@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send an email with the given subject and HTML body"""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))

    from_email = Email("your_verified_sender@email.com")
    to_email = To("recipient@email.com")

    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()

    response = sg.client.mail.send.post(request_body=mail)
    print("Email response", response.status_code)

    return {"status": "sent"}


INSTRUCTIONS = """
        You are a job assistant that sends job search results via email.

        Convert the job report into a clean, professional HTML email.

        Requirements:
        - Clear sections
        - Clickable job links
        - Clean formatting
        - Professional tone

        Subject line:
        "Top Job Matches for [User Query]"
    """

email_agent = Agent(
    name="JobEmailAgent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)

In [ ]:
from agents import Runner, trace, gen_trace_id
from planner_agent import planner_agent, WebSearchPlan, WebSearchItem
from search_agent import search_agent
from writer_agent import writer_agent, JobReport
from email_agent import email_agent

import asyncio


class JobSearchManager:

    async def run(self, query: str, send_email_flag: bool = False):
        trace_id = gen_trace_id()

        with trace("Job Search Trace", trace_id=trace_id):
            yield f"Trace: https://platform.openai.com/traces/trace?trace_id={trace_id}"

            yield "Planning job searches..."
            search_plan = await self.plan_searches(query)

            yield "Searching for jobs..."
            jobs = await self.perform_searches(search_plan)

            yield "Formatting results..."
            report = await self.write_report(query, jobs)

            if send_email_flag:
                yield "Sending email..."
                await self.send_email(report)

            yield "Done!"
            yield report.markdown_report

    async def plan_searches(self, query: str) -> WebSearchPlan:
        result = await Runner.run(planner_agent, f"User job query: {query}")
        return result.final_output_as(WebSearchPlan)

    async def perform_searches(self, search_plan: WebSearchPlan):
        tasks = [asyncio.create_task(self.search(item)) for item in search_plan.searches]

        all_jobs = []

        for task in asyncio.as_completed(tasks):
            result = await task
            if result:
                all_jobs.extend(result.jobs)

        return self.deduplicate_jobs(all_jobs)

    async def search(self, item: WebSearchItem):
        input_text = f"Job search query: {item.query}\nReason: {item.reason}"

        try:
            result = await Runner.run(search_agent, input_text)
            return result.final_output_as(type(result.final_output))
        except Exception as e:
            print(f"Search failed: {e}")
            return None

    def deduplicate_jobs(self, jobs):
        seen = set()
        unique_jobs = []

        for job in jobs:
            key = (job.title, job.company, job.location)
            if key not in seen:
                seen.add(key)
                unique_jobs.append(job)

        return unique_jobs

    async def write_report(self, query: str, jobs):
        input_text = f"""
        User query: {query}

        Job listings:
        {jobs}
        """

        result = await Runner.run(writer_agent, input_text)
        return result.final_output_as(JobReport)

    async def send_email(self, report: JobReport):
        await Runner.run(email_agent, report.markdown_report)

## Example usage

In [ ]:
import asyncio
from job_search_manager import JobSearchManager

async def main():
    manager = JobSearchManager()

    async for update in manager.run(
        query="Junior Data Analyst remote",
        send_email_flag=False
    ):
        print(update)

asyncio.run(main())